<a href="https://colab.research.google.com/github/tkratt/GARCH_vs_LSTM_OPTION_PRICING/blob/main/Merton_Jump_Diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Sample Code**

In [1]:

import numpy as np
from scipy.special import ndtr, gammaln
from scipy.stats import poisson


def merton_call(S, K, T, r, sigma, lam, jump_mean, jump_std, q=0.0,
                tol=1e-12):
    """

    Parameters
    ----------
    S : float
        Current underlying asset price.
    K : float or array
        Strike price(s).
    T : float
        Time to maturity in years (> 0).
    r : float
        Annual continuously compounded risk-free rate.
    sigma : float
        Annual diffusion volatility (decimal, e.g. 0.20).
    lam : float
        Annual Poisson jump intensity.
    jump_mean : float
        Mean of normally distributed log-jump size.
    jump_std : float
        Standard deviation of log-jump size.
    q : float, optional
        Continuous dividend yield (default 0).
        FX: q = foreign interest rate.
        Oil futures: use q = r.

    tol : float, optional
        Poisson tail tolerance for truncating the infinite sum.

    Returns
    -------
    float or numpy.ndarray
        European call option price(s).

    Notes
    -----
    Jump sizes are lognormally distributed:
        Y ~ N(jump_mean, jump_std^2)

    The risk-neutral jump compensator is:
        kappa_J = exp(jump_mean + 0.5*jump_std^2) - 1

    Put = Call - S*exp(-q*T) + K*exp(-r*T)
    """

    K = np.atleast_1d(np.asarray(K, dtype=float))

    if S <= 0 or np.any(K <= 0):
        raise ValueError("S and K must be positive.")

    if T <= 0 or sigma <= 0:
        raise ValueError("T and sigma must be positive.")

    if lam < 0 or jump_std < 0:
        raise ValueError("lam and jump_std must be non-negative.")

    if not 0 < tol < 1:
        raise ValueError("tol must be between 0 and 1.")

    # Expected percentage jump size
    kappa_J = np.exp(jump_mean + 0.5 * jump_std**2) - 1

    # Expected number of jumps over the option's lifetime
    intensity = lam * T

    # Truncation point for the Poisson summation
    if intensity == 0:
        n_max = 0
    else:
        n_max = int(poisson.ppf(1 - tol, intensity))

    n = np.arange(n_max + 1)

    # Poisson probabilities
    log_weights = (
        -intensity
        + n * np.log(intensity) if intensity > 0
        else np.zeros_like(n, dtype=float)
    )

    if intensity > 0:
        log_weights -= gammaln(n + 1)

    weights = np.exp(log_weights)

    # Conditional volatility for each number of jumps
    sigma_n = np.sqrt(sigma**2 + n * jump_std**2 / T)

    # Conditional adjusted spot prices
    S_n = S * np.exp(
        -lam * kappa_J * T
        + n * (jump_mean + 0.5 * jump_std**2)
    )

    # Black-Scholes price for each conditional jump count
    K_col = K[:, None]
    S_row = S_n[None, :]
    vol_row = sigma_n[None, :]

    sqrt_T = np.sqrt(T)

    d1 = (
        np.log(S_row / K_col)
        + (r - q + 0.5 * vol_row**2) * T
    ) / (vol_row * sqrt_T)

    d2 = d1 - vol_row * sqrt_T

    bs_calls = (
        S_row * np.exp(-q * T) * ndtr(d1)
        - K_col * np.exp(-r * T) * ndtr(d2)
    )

    # Poisson-weighted sum of conditional Black-Scholes prices
    price = bs_calls @ weights

    return float(price[0]) if price.size == 1 else price


def merton_put(S, K, T, r, sigma, lam, jump_mean, jump_std, q=0.0):
    K_arr = np.atleast_1d(np.asarray(K, dtype=float))

    call = merton_call(
        S, K_arr, T, r, sigma, lam,
        jump_mean, jump_std, q
    )

    put = call - S * np.exp(-q * T) + K_arr * np.exp(-r * T)

    return float(put[0]) if put.size == 1 else put

**Example Usage**

In [2]:
S = 100
K = np.array([80, 90, 100, 110, 120])

T = 1.0
r = 0.05
q = 0.0

sigma = 0.20
lam = 0.75

jump_mean = -0.10
jump_std = 0.25

calls = merton_call(
    S, K, T, r, sigma, lam,
    jump_mean, jump_std, q
)

puts = merton_put(
    S, K, T, r, sigma, lam,
    jump_mean, jump_std, q
)

print("Merton Jump-Diffusion Option Prices")
print("-----------------------------------")

for strike, call, put in zip(K, calls, puts):
    print(
        f"K = {strike:6.2f} | "
        f"Call = {call:8.4f} | "
        f"Put = {put:8.4f}"
    )

Merton Jump-Diffusion Option Prices
-----------------------------------
K =  80.00 | Call =  26.6916 | Put =   2.7900
K =  90.00 | Call =  19.6009 | Put =   5.2116
K = 100.00 | Call =  13.7658 | Put =   8.8887
K = 110.00 | Call =   9.3047 | Put =  13.9400
K = 120.00 | Call =   6.1218 | Put =  20.2694


**Validation Tests**

In [3]:
# Test 1: Zero jump intensity should reproduce Black-Scholes

from scipy.stats import norm

def black_scholes_call(S, K, T, r, sigma, q=0.0):
    d1 = (
        np.log(S / K) + (r - q + 0.5 * sigma**2) * T
    ) / (sigma * np.sqrt(T))

    d2 = d1 - sigma * np.sqrt(T)

    return (
        S * np.exp(-q*T) * norm.cdf(d1)
        - K * np.exp(-r*T) * norm.cdf(d2)
    )


merton_zero_jump = merton_call(
    100, 100, 1, 0.05, 0.20, 0.0, -0.10, 0.25
)

bs_price = black_scholes_call(100, 100, 1, 0.05, 0.20)

assert np.isclose(merton_zero_jump, bs_price, atol=1e-10)

print("Test 1 passed: Merton converges to Black-Scholes.")


# Test 2: Put-call parity

call = merton_call(100, 100, 1, 0.05, 0.20, 0.75, -0.10, 0.25)
put = merton_put(100, 100, 1, 0.05, 0.20, 0.75, -0.10, 0.25)

assert np.isclose(
    call - put,
    100 - 100 * np.exp(-0.05),
    atol=1e-10
)

print("Test 2 passed: Put-call parity holds.")


# Test 3: Prices should be positive

prices = merton_call(
    100, np.array([80, 90, 100, 110, 120]),
    1, 0.05, 0.20, 0.75, -0.10, 0.25
)

assert np.all(prices > 0)

print("Test 3 passed: All call prices are positive.")

Test 1 passed: Merton converges to Black-Scholes.
Test 2 passed: Put-call parity holds.
Test 3 passed: All call prices are positive.
